# Merge CLEAR2026 Evaluation Datasets

Standalone utility to merge processed PNG+SQLite datasets into one dataset folder. It keeps the same schema, rewrites relative `png_path` values under a source tag, and prefixes `run_id`/`image_id` so the merged output can be merged again later without accidental collisions.

In [1]:
from pathlib import Path, PurePosixPath
import shutil
import sqlite3
from datetime import datetime, timezone


SOURCES = [
    Path("/Users/andrewxu/Desktop/AI_Workspace/dataset/processed/405_realbeam_random_256"),
    Path("/Users/andrewxu/Desktop/AI_Workspace/dataset/processed/405_realbeam_raster_256"),
]
OUT_DIR = Path("/Users/andrewxu/Desktop/AI_Workspace/dataset/processed/405_realbeam_eval_256")
OVERWRITE = False
DB_NAME = "dataset.db"


def source_tag(index, source_dir):
    return f"src{index:02d}_{source_dir.name}"


def table_columns(conn, table):
    return [row[1] for row in conn.execute(f"PRAGMA table_info({table})")]


def insert_row(conn, table, row):
    cols = list(row.keys())
    placeholders = ", ".join(["?"] * len(cols))
    names = ", ".join(cols)
    conn.execute(
        f"INSERT INTO {table} ({names}) VALUES ({placeholders})",
        [row[col] for col in cols],
    )


def rel_path(value):
    return PurePosixPath(str(value).lstrip("/"))


def prepare_output():
    if OUT_DIR.exists():
        if not OVERWRITE:
            raise FileExistsError(f"Output exists: {OUT_DIR}. Set OVERWRITE=True to replace it.")
        shutil.rmtree(OUT_DIR)

    OUT_DIR.mkdir(parents=True)
    shutil.copy2(SOURCES[0] / DB_NAME, OUT_DIR / DB_NAME)

    with sqlite3.connect(OUT_DIR / DB_NAME) as conn:
        conn.execute("PRAGMA foreign_keys = OFF")
        conn.execute("DELETE FROM images")
        conn.execute("DELETE FROM run_info")
        conn.execute("DELETE FROM cameras")
        conn.commit()


def merge_source(out_conn, src_dir, tag):
    src_db = src_dir / DB_NAME
    if not src_db.exists():
        raise FileNotFoundError(src_db)

    with sqlite3.connect(src_db) as src_conn:
        src_conn.row_factory = sqlite3.Row

        for row in src_conn.execute("SELECT * FROM cameras"):
            data = dict(row)
            cols = list(data.keys())
            out_conn.execute(
                f"INSERT OR IGNORE INTO cameras ({', '.join(cols)}) VALUES ({', '.join(['?'] * len(cols))})",
                [data[col] for col in cols],
            )

        for row in src_conn.execute("SELECT key, value FROM run_info"):
            out_conn.execute(
                "INSERT OR REPLACE INTO run_info (key, value) VALUES (?, ?)",
                (f"{tag}.{row['key']}", row["value"]),
            )

        out_conn.execute(
            "INSERT OR REPLACE INTO run_info (key, value) VALUES (?, ?)",
            (f"merge_source.{tag}", str(src_dir)),
        )

        copied = 0
        for row in src_conn.execute("SELECT * FROM images ORDER BY camera, sample_id, image_id"):
            data = dict(row)
            old_png = rel_path(data["png_path"])
            new_png = PurePosixPath(tag) / old_png

            src_png = src_dir / old_png
            dst_png = OUT_DIR / new_png
            if not src_png.exists():
                raise FileNotFoundError(src_png)
            dst_png.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_png, dst_png)

            data["image_id"] = f"{tag}::{data['image_id']}"
            if data.get("run_id") is not None:
                data["run_id"] = f"{tag}::{data['run_id']}"
            data["png_path"] = new_png.as_posix()
            insert_row(out_conn, "images", data)
            copied += 1

    return copied


def count_pairs(conn, camera_in="cam1", camera_out="cam2"):
    return conn.execute(
        """
        SELECT COUNT(*)
        FROM images a
        JOIN images b
          ON a.sample_id = b.sample_id
         AND a.run_id = b.run_id
        WHERE a.camera = ?
          AND b.camera = ?
          AND a.png_path IS NOT NULL
          AND b.png_path IS NOT NULL
        """,
        (camera_in, camera_out),
    ).fetchone()[0]


prepare_output()

with sqlite3.connect(OUT_DIR / DB_NAME) as out_conn:
    out_conn.execute("PRAGMA foreign_keys = OFF")
    counts = {}
    for index, src in enumerate(SOURCES):
        tag = source_tag(index, src)
        counts[tag] = merge_source(out_conn, src, tag)

    out_conn.execute(
        "INSERT OR REPLACE INTO run_info (key, value) VALUES (?, ?)",
        ("merged_created_utc", datetime.now(timezone.utc).isoformat()),
    )
    out_conn.execute(
        "CREATE INDEX IF NOT EXISTS ix_images_camera_run_sample ON images(camera, run_id, sample_id)"
    )
    out_conn.commit()

    total = out_conn.execute("SELECT COUNT(*) FROM images").fetchone()[0]
    by_camera = out_conn.execute(
        "SELECT camera, COUNT(*) FROM images GROUP BY camera ORDER BY camera"
    ).fetchall()
    cam1_cam2_pairs = count_pairs(out_conn, "cam1", "cam2")
    cam3_cam2_pairs = count_pairs(out_conn, "cam3", "cam2")

print("merged dataset:", OUT_DIR)
print("source rows:", counts)
print("total image rows:", total)
print("rows by camera:", dict(by_camera))
print("cam1->cam2 pairs:", cam1_cam2_pairs)
print("cam3->cam2 pairs:", cam3_cam2_pairs)


merged dataset: /Users/andrewxu/Desktop/AI_Workspace/dataset/processed/405_realbeam_eval_256
source rows: {'src00_405_realbeam_random_256': 3004, 'src01_405_realbeam_raster_256': 1893}
total image rows: 4897
rows by camera: {'cam1': 1633, 'cam2': 1632, 'cam3': 1632}
cam1->cam2 pairs: 1631
cam3->cam2 pairs: 1630
